# ANLP Assignment 2: Colab runner (all parts)

The course git server is not reachable from Colab, so the code travels via Google Drive:
on your machine run `git archive --format=zip -o colab_code.zip HEAD` and upload
`colab_code.zip` to `My Drive/anlp_a2/`.

Secrets (key icon, notebook access on): `WANDB_API_KEY`, `HF_TOKEN` (write).
Checkpoints/results are written to Drive, so a disconnect loses nothing: rerun the
setup cells, then the training cell resumes from the last checkpoint.

## Setup (run every session)

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/anlp_a2'
os.makedirs(DRIVE, exist_ok=True)
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
ZIP = f'{DRIVE}/colab_code.zip'
assert os.path.exists(ZIP), f'upload colab_code.zip to {DRIVE} first'
!rm -rf /content/repo && mkdir -p /content/repo
!unzip -q -o {ZIP} -d /content/repo
%cd /content/repo

In [ ]:
!pip install -q sacrebleu wandb
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Part 1: MoE ablations

In [ ]:
!python -m src.part1.data --out /content/data/part1

In [ ]:
VARIANTS = ['1_dense', '2_moe_4e_top1', '3_moe_4e_top2', '4_moe_shared1_3e_top1', '5_moe_4e_top2_active']
for v in VARIANTS:
    ckpt, res = f'{DRIVE}/checkpoints/part1/{v}', f'{DRIVE}/results/part1/{v}'
    print(f'
========== {v} ==========')
    !python -m src.part1.train --config configs/part1/{v}.yaml --data /content/data/part1 --out {ckpt}
    if not os.path.exists(f'{ckpt}/model.pt'):
        print(f'training {v} did not finish, stopping'); break
    !python -m src.part1.evaluate --ckpt {ckpt}/model.pt --data /content/data/part1 --out {res}
    !python -m src.hf_upload --ckpt {ckpt}/model.pt --results {res} --repo anlp-a2-part1-{v}

In [ ]:
!python -m src.part1.analyze_routing --ckpt-root {DRIVE}/checkpoints/part1 --data /content/data/part1 --out {DRIVE}/results/part1/routing
!python -m src.part1.plots --results {DRIVE}/results/part1 --wandb-path <wandb-entity>/anlp-a2-part1

## Part 2: optimizers

In [ ]:
!python -m src.part2.data --out /content/data/part2

In [ ]:
OPTS = ['adamw', 'mars', 'lion', 'muon']   # split across two sessions to run in parallel
for o in OPTS:
    ckpt = f'{DRIVE}/checkpoints/part2/{o}'
    print(f'
========== {o} ==========')
    !python -m src.part2.train --optimizer {o} --data /content/data/part2 --out {ckpt}
    if not os.path.exists(f'{ckpt}/model.pt'):
        print(f'{o} did not finish, stopping'); break
    !python -m src.hf_upload --ckpt {ckpt}/model.pt --results {ckpt} --repo anlp-a2-part2-{o}

Optional (bonus): short LR sweeps on 0.1x of the dataset.

In [ ]:
for o, lrs in {'adamw': [5e-4, 1e-3, 2e-3], 'mars': [5e-4, 1e-3, 2e-3], 'lion': [5e-5, 1e-4, 2e-4], 'muon': [5e-4, 1e-3, 2e-3]}.items():
    for lr in lrs:
        name = f'sweep_{o}_lr{lr:g}'
        !python -m src.part2.train --optimizer {o} --lr {lr} --data-frac 0.1 --run-name {name} --data /content/data/part2 --out {DRIVE}/checkpoints/part2/{name}

In [ ]:
!python -m src.part2.plots --root {DRIVE}/checkpoints/part2 --out {DRIVE}/results/part2

## Part 3: decoding strategies (Pythia-160M, no training)

In [ ]:
!python -m src.part3.run --n 1000 --out {DRIVE}/results/part3